# Parametric boundaries

**What this shows:** how to prescribe incoming waves by their parameters with
`BOUNDSPEC`: the spectral shape, where the boundary is, and constant, varying or
time-dependent values.

**Prerequisites:** [Tutorial 4: Wave boundaries](../tutorial/04_wave_boundaries.ipynb).

**You will learn:**

- how `SHAPESPEC` sets the spectral shape, the period and the directional spreading
- how to apply a boundary on a side or along a segment
- how to vary the parameters along the boundary and in time (TPAR files)

**Data used:** none; a TPAR file is created in the notebook.

## Setup

In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from rompy_swan.components.boundary import BOUNDSPEC
from rompy_swan.subcomponents.base import XY
from rompy_swan.subcomponents.boundary import (
    CONSTANTFILE,
    CONSTANTPAR,
    SEGMENT,
    SIDE,
    VARIABLEPAR,
)
from rompy_swan.subcomponents.spectrum import GAUSS, JONSWAP, PM, SHAPESPEC, TMA

OUT_DIR = Path("_output") / "boundaries_parametric"
shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir(parents=True)

## 1. The spectral shape

`SHAPESPEC` becomes SWAN's `BOUND SHAPESPEC` command:

- `shape`: `JONSWAP(gamma)` (default, `gamma=3.3`), `PM` (Pierson-Moskowitz, fully
  developed sea), `GAUSS(sigfr)` (narrow swell), `TMA(gamma, d)` (JONSWAP in finite
  depth `d`) or `BIN` (all energy in one frequency bin);
- `per_type`: whether the period is the `peak` or the `mean` period;
- `dspr_type`: whether the spreading is given in `degrees` (directional standard
  deviation) or as the `power` m of cos^m.

In [2]:
for shape in [JONSWAP(gamma=3.3), PM(), GAUSS(sigfr=0.01), TMA(gamma=3.3, d=20.0)]:
    print(SHAPESPEC(shape=shape, per_type="peak", dspr_type="degrees").render())

BOUND SHAPESPEC JONSWAP gamma=3.3 PEAK DSPR DEGREES
BOUND SHAPESPEC PM PEAK DSPR DEGREES
BOUND SHAPESPEC GAUSS sigfr=0.01 PEAK DSPR DEGREES
BOUND SHAPESPEC TMA gamma=3.3 d=20.0 PEAK DSPR DEGREES


## 2. Where the boundary is

`SIDE` applies the boundary to a whole side of a regular grid. `direction` says which
way positions along the side are counted, which matters for varying parameters.

In [3]:
jonswap = SHAPESPEC(shape=JONSWAP(gamma=3.3), per_type="peak", dspr_type="degrees")
swell = CONSTANTPAR(hs=2.0, per=12.0, dir=240.0, dd=20.0)

print(BOUNDSPEC(shapespec=jonswap, location=SIDE(side="west", direction="ccw"), data=swell).render())

BOUND SHAPESPEC JONSWAP gamma=3.3 PEAK DSPR DEGREES
BOUNDSPEC SIDE WEST CCW CONSTANT PAR hs=2.0 per=12.0 dir=240.0 dd=20.0


`SEGMENT` applies it along a line given by points (`XY`, in the problem coordinates)
or by grid indices, for example the deep-water part of a side:

In [4]:
segment = SEGMENT(points=XY(x=[114.5, 114.5], y=[-32.8, -32.0]))
print(BOUNDSPEC(shapespec=jonswap, location=segment, data=swell).render())

BOUND SHAPESPEC JONSWAP gamma=3.3 PEAK DSPR DEGREES
BOUNDSPEC SEGMENT XY  &
    114.50000000 -32.80000000 &
    114.50000000 -32.00000000 &
    CONSTANT PAR hs=2.0 per=12.0 dir=240.0 dd=20.0


## 3. Parameters that vary along the boundary

`VARIABLEPAR` gives values at distances `len` along the side or segment, measured
from its start in the counting direction. SWAN interpolates between them. Here the
waves grow northwards along the western side, 1.3° long.

In [5]:
variable = VARIABLEPAR(
    hs=[1.5, 2.0, 2.5],
    per=[11.0, 12.0, 12.0],
    dir=[235.0, 240.0, 245.0],
    dd=[20.0, 20.0, 20.0],
    len=[0.0, 0.65, 1.3],
)
print(
    BOUNDSPEC(
        shapespec=jonswap, location=SIDE(side="west", direction="clockwise"), data=variable
    ).render()
)

BOUND SHAPESPEC JONSWAP gamma=3.3 PEAK DSPR DEGREES
BOUNDSPEC SIDE WEST CLOCKWISE VARIABLE PAR & &
    	len=0.0 hs=1.5 per=11.0 dir=235.0 dd=20.0 & &
    	len=0.65 hs=2.0 per=12.0 dir=240.0 dd=20.0 & &
    	len=1.3 hs=2.5 per=12.0 dir=245.0 dd=20.0


## 4. Parameters that vary in time: TPAR files

A TPAR file holds a time series of wave height, period, direction and spreading, for
example from a buoy or a wave model output point. `write_tpar` writes one from a
pandas DataFrame indexed by time, with the columns in that order.

In [6]:
from rompy_swan.boundary import write_tpar

times = pd.date_range("2023-01-01", periods=7, freq="1h")
buoy = pd.DataFrame(
    {
        "hs": np.linspace(2.0, 3.0, len(times)),
        "tp": 12.0,
        "dir": np.linspace(240.0, 230.0, len(times)),
        "dspr": 20.0,
    },
    index=times,
)
write_tpar(buoy, OUT_DIR / "buoy.tpar")
print((OUT_DIR / "buoy.tpar").read_text())

TPAR
20230101.000000 2.00 12.00 240.00 20.00
20230101.010000 2.17 12.00 238.33 20.00
20230101.020000 2.33 12.00 236.67 20.00
20230101.030000 2.50 12.00 235.00 20.00
20230101.040000 2.67 12.00 233.33 20.00
20230101.050000 2.83 12.00 231.67 20.00
20230101.060000 3.00 12.00 230.00 20.00



`CONSTANTFILE` uses the same file along the whole side:

In [7]:
tpar_boundary = BOUNDSPEC(
    shapespec=jonswap,
    location=SIDE(side="west", direction="ccw"),
    data=CONSTANTFILE(fname="buoy.tpar", seq=1),
)
print(tpar_boundary.render())

BOUND SHAPESPEC JONSWAP gamma=3.3 PEAK DSPR DEGREES
BOUNDSPEC SIDE WEST CCW CONSTANT FILE fname='buoy.tpar' seq=1


`VARIABLEFILE` takes several files at distances along the boundary, like
`VARIABLEPAR`. TPAR files have times, so the model needs `MODE NONSTATIONARY`, and
the file must be in the workspace: copy it there, or let a data-driven boundary write
it ([Boundaries from spectra](boundaries_from_spectra.ipynb)).

## Summary

- `SHAPESPEC` sets the spectral shape, period type and spreading units.
- `SIDE` covers a whole side; `SEGMENT` a line of points or grid indices.
- `CONSTANTPAR` and `VARIABLEPAR` give fixed values; `CONSTANTFILE` and
  `VARIABLEFILE` give time series from TPAR files.
- A `BOUNDSPEC` covers one side or segment; for waves on every open side, use
  spectra ([Tutorial 4](../tutorial/04_wave_boundaries.ipynb)).